# Demand Modelling

This notebook creates generates elasticity matrices for three consumer groups

## Importing the Data and Dependencies

Importing required packages:

In [1]:
from google.colab import drive
import pandas as pd
import numpy as np

Mounting to Google Drive:

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


## Modelling Price Elasticities

Creating functions to produce elasticity matrices. Cross-price elasticities are assigned using a symmetric Gaussian kernel to ensuring that the elements of each row sum to zero.



In [3]:
T = 24

def circ_dist(i, j, T=24):
    d = abs(i - j)
    return min(d, T - d)

def elasticity_matrix(alpha: float, sigma: float) -> np.ndarray:

    if alpha <= 0:
        raise ValueError("alpha must be > 0.")
    if sigma is None or sigma <= 0:
        raise ValueError("sigma must be > 0 for a Gaussian kernel.")

    e = np.zeros((T, T), dtype=float)

    W = np.zeros((T, T), dtype=float)
    for i in range(T):
        for j in range(T):
            if i == j:
                continue
            d = circ_dist(i, j, T)
            W[i, j] = np.exp(-(d * d) / (2.0 * sigma * sigma))

    t = np.full(T, alpha, dtype=float)
    row_w = W.sum(axis=1)
    u = np.where(row_w > 0, t / (row_w + 1e-16), 0.0)
    for _ in range(10000):
        Wu = W @ u
        u_new = np.zeros_like(u)
        mask = Wu > 0
        u_new[mask] = t[mask] / Wu[mask]
        if np.max(np.abs(u_new - u)) < 1e-10:
            u = u_new
            break
        u = u_new

    Eoff = (u[:, None] * W) * u[None, :]


    e += Eoff
    np.fill_diagonal(e, -alpha)

    return e

Specifying three consumer groups

In [4]:
IS = elasticity_matrix(alpha=0.002, sigma=1.5)
SC = elasticity_matrix(alpha=0.020, sigma=2.0)
HS = elasticity_matrix(alpha=0.060, sigma=4.0)

## Exporting Data

Exporting elasticity matrices:

In [5]:
pd.DataFrame(IS).to_csv("/content/drive/MyDrive/ERP/Processed_Data/IS.csv")
pd.DataFrame(SC).to_csv("/content/drive/MyDrive/ERP/Processed_Data/SC.csv")
pd.DataFrame(HS).to_csv("/content/drive/MyDrive/ERP/Processed_Data/HS.csv")